# 05 — Demographic segmentation of users

Demographic and geographic clustering, without behavioural variables. The aim is to obtain
segments assignable to any new user from the very first contact, with no need for an
interaction history.

Difference with `06_segmentation.ipynb`:
- M1 uses demographics + behaviour → segments informative of the current customer
- This notebook uses demographics only → operational segments for cold-start in production

Features used (10): `age`, `gender`, `labor_status`, `civil_status`, `tiene_coche`,
`size_hogar`, `num_room`, `ipa_class`, `mun_type`, `distance_type`

Dimensionality reduction evaluated: PCA and Autoencoder (the one with the best separation is chosen).
Clustering algorithms evaluated: KMeans and HDBSCAN.
Output: `data/processed/users_demo_segments.csv` — columns `id_user`, `demo_cluster`, `demo_cluster_label`

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # exact reproducibility of the autoencoder
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score
import hdbscan
import umap

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Add src/ to the path and reuse the shared helper (same pattern as 01-04)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import find_project_root  # noqa: E402

ROOT_PATH      = find_project_root()
DATA_PATH      = ROOT_PATH / "data"
PROCESSED_PATH = DATA_PATH / "processed"

np.random.seed(42)
print("ROOT:", ROOT_PATH)

## 1 · Loading and encoding of demographic features

In [ ]:
users = pd.read_csv(PROCESSED_PATH / "users.csv", dtype={"cp_num": str})
print("users:", users.shape)
display(users.head(2))

In [ ]:
df = users.copy()

df["gender_enc"]       = (df["gender"] == "H").astype(int)
df["labor_status_enc"] = df["labor_status"].map({"employed": 2, "unemployed": 1, "inactive": 0}).fillna(0).astype(int)
df["civil_status_enc"] = df["civil_status"].map({"casado": 3, "soltero": 0, "divorciado": 1, "viudo": 2}).fillna(0).astype(int)
df["tiene_coche_enc"]  = df["tiene_coche"].astype(int)

def parse_hogar(s):
    if pd.isna(s): return 3
    s = str(s).strip()
    if s.startswith("1"):   return 1
    elif s.startswith("2"): return 2
    elif s.startswith("3"): return 3
    elif s.startswith("4"): return 4
    else:                   return 5

df["size_hogar_enc"] = df["size_hogar"].apply(parse_hogar)
df["num_room_enc"]   = df["num_room"].map({"menos_3_hab": 1, "3_a_6_hab": 2, "7_mas_hab": 3}).fillna(2).astype(int)

def edad_a_grupo(edad):
    # Age bands (same as the M2 model): 18-24, 25-34, 35-44, 45-54, 55-64, 65+
    if pd.isna(edad): return -1
    for i, lim in enumerate([25, 35, 45, 55, 65]):
        if edad < lim: return i
    return 5
df["age_cat"] = df["age"].apply(edad_a_grupo)

DEMO_FEATS = ["age_cat", "gender_enc", "labor_status_enc", "civil_status_enc",
              "tiene_coche_enc", "size_hogar_enc", "num_room_enc",
              "ipa_class", "mun_type", "distance_type"]

X_raw = df[DEMO_FEATS].fillna(-1).values

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f"Feature matrix: {X.shape}")
print(f"Nulls in original features: {df[DEMO_FEATS].isna().sum().sum()}")
pd.DataFrame(X, columns=DEMO_FEATS).describe().round(2)

## 2 · Exploratory analysis of features

In [ ]:
# Distributions of demographic features
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

labels = {
    "age_cat":          "Age (band 0-5)",
    "gender_enc":       "Gender (1=M)",
    "labor_status_enc": "Labour status (0=inac, 2=empl)",
    "civil_status_enc": "Marital status",
    "tiene_coche_enc":  "Owns a car",
    "size_hogar_enc":   "Household size",
    "num_room_enc":     "No. of rooms",
    "ipa_class":        "IPA class (0-4)",
    "mun_type":         "Municipality type (0-5)",
    "distance_type":    "Dist. to capital (0-4)",
}

for ax, feat in zip(axes, DEMO_FEATS):
    vals = df[feat].dropna()
    if vals.nunique() <= 6:
        vals.value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    else:
        ax.hist(vals, bins=30, color="steelblue", edgecolor="white")
    ax.set_title(labels[feat], fontsize=9)
    ax.set_xlabel("")

plt.suptitle("Distribution of demographic features", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation between features
corr = df[DEMO_FEATS].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title("Correlation between demographic features")
plt.tight_layout()
plt.show()

# Pairs with high correlation
high_corr = [(DEMO_FEATS[i], DEMO_FEATS[j], corr.iloc[i,j])
             for i in range(len(DEMO_FEATS)) for j in range(i+1, len(DEMO_FEATS))
             if abs(corr.iloc[i,j]) > 0.3]
if high_corr:
    print("Correlations > 0.3:")
    for a, b, c in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f"  {a} × {b}: {c:.3f}")
else:
    print("No correlations > 0.3 between features.")

## 3 · UMAP projection

Reduction to 2D to visualise the latent structure of the demographic population.

In [ ]:
reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                    random_state=42, n_jobs=1)
embedding = reducer.fit_transform(X)
df["umap_x"] = embedding[:, 0]
df["umap_y"] = embedding[:, 1]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for ax, feat in zip(axes, DEMO_FEATS):
    sc = ax.scatter(df["umap_x"], df["umap_y"], c=df[feat], cmap="viridis",
                    s=1, alpha=0.4, rasterized=True)
    plt.colorbar(sc, ax=ax, shrink=0.8)
    ax.set_title(labels[feat], fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("UMAP coloured by demographic feature", fontsize=13)
plt.tight_layout()
plt.show()

## 3.5 · Dimensionality reduction (PCA)

PCA is applied to reduce the redundancy between correlated features. Selection
criterion: minimum number of components that explain 80% of the total variance.

In [ ]:
# Components needed for 80% of variance
pca_full = PCA(random_state=42)
pca_full.fit(X)

cumvar = pca_full.explained_variance_ratio_.cumsum()
n_pca  = int(np.argmax(cumvar >= 0.80) + 1)
print(f"Components needed to explain 80% of variance: {n_pca}")
print(f"Cumulative variance with {n_pca} components: {cumvar[n_pca-1]:.1%}")

# Explained variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_ * 100, color="steelblue")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Explained variance (%)")
axes[0].set_title("Variance explained by each component")

axes[1].plot(range(1, len(cumvar) + 1), cumvar * 100, marker="o", color="steelblue")
axes[1].axhline(80, color="crimson",  linestyle="--", linewidth=1.2, label="80%")
axes[1].axhline(90, color="orange",   linestyle="--", linewidth=1.2, label="90%")
axes[1].axvline(n_pca, color="crimson", linestyle=":", linewidth=1.5)
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance (%)")
axes[1].set_title(f"Cumulative variance — {n_pca} components needed for 80%")
axes[1].legend()

plt.tight_layout()
plt.show()

# Apply PCA with n_pca components
pca  = PCA(n_components=n_pca, random_state=42)
X_pca = pca.fit_transform(X)
print(f"\nPCA space: {X.shape} → {X_pca.shape}")

In [ ]:
# UMAP coloured by each PCA component (base map = 2D UMAP from section 3)
n_cols  = min(n_pca, 5)
n_rows  = int(np.ceil(n_pca / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = np.array(axes).flatten()

for i in range(n_pca):
    sc = axes[i].scatter(df["umap_x"], df["umap_y"], c=X_pca[:, i],
                         cmap="RdBu_r", s=1, alpha=0.4, rasterized=True)
    plt.colorbar(sc, ax=axes[i], shrink=0.8)
    axes[i].set_title(f"PC {i+1}  ({pca.explained_variance_ratio_[i]:.1%} var)", fontsize=9)
    axes[i].set_xticks([]); axes[i].set_yticks([])

for j in range(n_pca, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f"UMAP coloured by the {n_pca} PCA components (80% variance)", fontsize=13)
plt.tight_layout()
plt.show()

## 3.6 · Dimensionality reduction (Autoencoder)

Non-linear reduction of the feature space via a neural network that compresses the input
to a latent space and reconstructs it, minimising the reconstruction error (MSE).

Architecture: 10 → 64 → 32 → 4D latent → 32 → 64 → 10

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

# Full determinism: fix seeds and deterministic operations so that 05 and 05b
# reconstruct the same X_ae latent space on any run.
tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

INPUT_DIM  = X.shape[1]
LATENT_DIM = 4

# Encoder: 10D -> 4D
inp    = Input(shape=(INPUT_DIM,))
h      = Dense(64, activation="relu")(inp)
h      = BatchNormalization()(h)
h      = Dense(32, activation="relu")(h)
latent = Dense(LATENT_DIM, activation="linear", name="latent")(h)

# Decoder: 4D -> 10D
h   = Dense(32, activation="relu")(latent)
h   = BatchNormalization()(h)
h   = Dense(64, activation="relu")(h)
out = Dense(INPUT_DIM, activation="linear")(h)

autoencoder = Model(inp, out)    # full network (training)
encoder     = Model(inp, latent) # compression part (clustering)

autoencoder.compile(optimizer="adam", loss="mse")

# Training with EarlyStopping on val_loss
history = autoencoder.fit(
    X, X,
    epochs           = 100,
    batch_size       = 256,
    validation_split = 0.1,
    callbacks        = [EarlyStopping(monitor="val_loss", patience=10,
                                      restore_best_weights=True)],
    verbose          = 0,
)

# Latent space embedding
X_ae = encoder.predict(X, verbose=0)

print(f"AE space: {X.shape} → {X_ae.shape}")
print(f"Epochs trained: {len(history.history['loss'])}")
print(f"Reconstruction error (train): {history.history['loss'][-1]:.4f}")
print(f"Reconstruction error (val):   {history.history['val_loss'][-1]:.4f}")

# Training curve
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(history.history["loss"],     label="Train", color="steelblue")
ax.plot(history.history["val_loss"], label="Val",   color="tomato", linestyle="--")
ax.set_xlabel("Epochs")
ax.set_ylabel("MSE error")
ax.set_title("Autoencoder — training curve")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualisation: UMAP coloured by each AE latent dimension ─────────────────
fig, axes = plt.subplots(1, LATENT_DIM, figsize=(4 * LATENT_DIM, 4))

for i, ax in enumerate(axes):
    sc = ax.scatter(df["umap_x"], df["umap_y"], c=X_ae[:, i],
                    cmap="RdBu_r", s=1, alpha=0.4, rasterized=True)
    plt.colorbar(sc, ax=ax, shrink=0.8)
    ax.set_title(f"Latent dim {i+1}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f"UMAP coloured by the {LATENT_DIM} dimensions of the Autoencoder latent space",
             fontsize=12)
plt.tight_layout()
plt.show()

## 3.7 · PCA vs Autoencoder comparison

The separation obtained in each space is compared via the Silhouette Score with k=2 (higher
values indicate greater separation between groups). Spaces compared:
- `X` — scaled original features (no reduction)
- `X_pca` — PCA space (80% variance)
- `X_ae` — Autoencoder latent space (4D)

The space with the highest silhouette is selected for clustering.

In [ ]:
# ── Silhouette k=2 in the three spaces ───────────────────────────────────────
sil_x_k2   = silhouette_score(X,     KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X),
                               sample_size=5000, random_state=42)
sil_pca_k2 = silhouette_score(X_pca, KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_pca),
                               sample_size=5000, random_state=42)
sil_ae_k2  = silhouette_score(X_ae,  KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_ae),
                               sample_size=5000, random_state=42)

# Summary table
df_comp_emb = pd.DataFrame({
    "Espacio":           ["Original (X)", f"PCA {n_pca}D (X_pca)", f"Autoencoder {LATENT_DIM}D (X_ae)"],
    "Dimensiones":       [X.shape[1], X_pca.shape[1], X_ae.shape[1]],
    "Silhouette k=2 ↑":  [round(sil_x_k2, 4), round(sil_pca_k2, 4), round(sil_ae_k2, 4)],
})
print("Comparison of embedding spaces:")
display(df_comp_emb)

# ── Automatic selection of the best space ─────────────────────────────────────
best_sil = max(sil_x_k2, sil_pca_k2, sil_ae_k2)
if best_sil == sil_ae_k2:
    X_cluster, X_cluster_name = X_ae,  f"Autoencoder {LATENT_DIM}D"
elif best_sil == sil_pca_k2:
    X_cluster, X_cluster_name = X_pca, f"PCA {n_pca}D"
else:
    X_cluster, X_cluster_name = X,     "Original"

AE_USED  = X_cluster is X_ae
PCA_USED = X_cluster is X_pca

print(f"\n→ Space selected for clustering: {X_cluster_name}  "
      f"(silhouette={best_sil:.4f})")

# ── Comparison chart ──────────────────────────────────────────────────────────
espacios = ["Original", f"PCA {n_pca}D", f"AE {LATENT_DIM}D"]
valores  = [sil_x_k2, sil_pca_k2, sil_ae_k2]
colores  = ["#aec6cf", "#aec6cf", "#aec6cf"]
colores[valores.index(best_sil)] = "#e05c4a"   # red for the winner

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(espacios, valores, color=colores, edgecolor="white", width=0.5)
ax.set_ylim(0, max(valores) * 1.25)
ax.set_ylabel("Silhouette Score k=2")
ax.set_title("Comparison of dimensionality reduction methods\n(higher silhouette = better separation)")

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## 4 · KMeans — selection of the optimal k

Four metrics are evaluated to choose the number of clusters:

| Metric | Interpretation | Optimum |
|---------|---------------|--------|
| **Silhouette** | Internal cohesion vs separation between clusters | Maximum |
| **Inertia (Elbow)** | Sum of intra-cluster distances | Elbow of the curve |
| **Calinski-Harabasz** | Ratio of inter/intra-cluster variance | Maximum |
| **Davies-Bouldin** | Mean similarity between pairs of clusters | Minimum |

In [ ]:
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

K_RANGE       = range(2, 13)
inertias      = []
sil_scores_km = []
ch_scores     = []
db_scores     = []

for k in K_RANGE:
    km       = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_cluster, labels_k, sample_size=5000, random_state=42)
    ch  = calinski_harabasz_score(X_cluster, labels_k)
    db  = davies_bouldin_score(X_cluster, labels_k)
    sil_scores_km.append(sil)
    ch_scores.append(ch)
    db_scores.append(db)
    print(f"  k={k:2d}  inertia={km.inertia_:9.1f}  silhouette={sil:.4f}  CH={ch:8.1f}  DB={db:.4f}")

best_k_sil = list(K_RANGE)[np.argmax(sil_scores_km)]
best_k_ch  = list(K_RANGE)[np.argmax(ch_scores)]
best_k_db  = list(K_RANGE)[np.argmin(db_scores)]
print(f"\noptimal k by Silhouette: {best_k_sil}")
print(f"optimal k by Calinski-Harabasz: {best_k_ch}")
print(f"optimal k by Davies-Bouldin: {best_k_db}")

# Simple majority vote
from collections import Counter
vote_k = Counter([best_k_sil, best_k_ch, best_k_db])
best_k_km = vote_k.most_common(1)[0][0]
# The silhouette and Calinski-Harabasz grow monotonically with k (up to the top of the
# range), a sign that the demographic space is almost a continuum with no natural groups:
# their argmax is not reliable. The decisive criterion is the STABILITY of the clustering (ARI by
# resampling, notebook 05b): k=10 is a local maximum of ARI (0.977, well above
# its neighbours k=9 and k=11) with high cohesion. k=10 is adopted.
best_k_km = 10  # see 05b: with age_cat the silhouette is almost monotonic and the ARI is high across the whole range
# (0.96-1.0); k=10 is kept for pipeline coherence and interpretable granularity (ARI 0.984)
print(f"\nk adopted (ARI stability, see 05b): {best_k_km}")

# Visualisation of the 4 metrics
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
ks = list(K_RANGE)

axes[0,0].plot(ks, inertias, marker="o", color="steelblue")
axes[0,0].set_title("Inertia (Elbow)"); axes[0,0].set_xlabel("k")

axes[0,1].plot(ks, sil_scores_km, marker="o", color="tomato")
axes[0,1].axvline(best_k_sil, color="crimson", linestyle="--", linewidth=1.2, label=f"k={best_k_sil}")
axes[0,1].set_title("Silhouette (↑ better)"); axes[0,1].set_xlabel("k"); axes[0,1].legend()

axes[1,0].plot(ks, ch_scores, marker="o", color="seagreen")
axes[1,0].axvline(best_k_ch, color="darkgreen", linestyle="--", linewidth=1.2, label=f"k={best_k_ch}")
axes[1,0].set_title("Calinski-Harabasz (↑ better)"); axes[1,0].set_xlabel("k"); axes[1,0].legend()

axes[1,1].plot(ks, db_scores, marker="o", color="orange")
axes[1,1].axvline(best_k_db, color="darkorange", linestyle="--", linewidth=1.2, label=f"k={best_k_db}")
axes[1,1].set_title("Davies-Bouldin (↓ better)"); axes[1,1].set_xlabel("k"); axes[1,1].legend()

plt.suptitle(f"KMeans — k selection metrics  [{X_cluster_name}]", fontsize=13)
plt.tight_layout()
plt.show()

## 5 · HDBSCAN

Density-based clustering — automatically detects the number of clusters
and labels non-assignable points as noise. Does not require specifying k.

In [ ]:
# Exploration of min_cluster_size
print("HDBSCAN — exploration of min_cluster_size:")
hdb_results = []
for mcs in [100, 200, 500, 1000, 2000]:
    hdb = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=10,
                          core_dist_n_jobs=-1, prediction_data=True)
    lbl = hdb.fit_predict(X_cluster)
    n_clusters = len(set(lbl)) - (1 if -1 in lbl else 0)
    noise_pct  = (lbl == -1).mean()
    if n_clusters > 1:
        mask = lbl != -1
        sil  = silhouette_score(X_cluster[mask], lbl[mask], sample_size=5000, random_state=42)
    else:
        sil = 0
    hdb_results.append({"min_cluster_size": mcs, "n_clusters": n_clusters,
                        "noise_pct": noise_pct, "silhouette": sil})
    print(f"  mcs={mcs:5d} → clusters={n_clusters}  noise={noise_pct:.1%}  sil={sil:.4f}")

df_hdb = pd.DataFrame(hdb_results)
display(df_hdb)

# Best configuration
best_mcs = int(df_hdb.loc[df_hdb["silhouette"].idxmax(), "min_cluster_size"])
hdb_final = hdbscan.HDBSCAN(min_cluster_size=best_mcs, min_samples=10,
                             core_dist_n_jobs=-1, prediction_data=True)
labels_hdb = hdb_final.fit_predict(X_cluster)
n_hdb      = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
noise_hdb  = (labels_hdb == -1).mean()
print(f"\nFinal HDBSCAN (mcs={best_mcs}): {n_hdb} clusters, {noise_hdb:.1%} noise")

## 6 · Algorithm comparison and final selection

KMeans (optimal k by consensus) and HDBSCAN are compared in the `X_cluster` space.

In [ ]:
# KMeans with the k chosen by consensus
km_comp  = KMeans(n_clusters=best_k_km, random_state=42, n_init=10)
lbl_km   = km_comp.fit_predict(X_cluster)
sil_km   = silhouette_score(X_cluster, lbl_km, sample_size=5000, random_state=42)
ch_km    = calinski_harabasz_score(X_cluster, lbl_km)
db_km    = davies_bouldin_score(X_cluster, lbl_km)

# HDBSCAN silhouette (excluding noise)
mask_hdb = labels_hdb != -1
sil_hdb  = silhouette_score(X_cluster[mask_hdb], labels_hdb[mask_hdb],
                             sample_size=5000, random_state=42) if mask_hdb.sum() > 1 else 0
ch_hdb   = calinski_harabasz_score(X_cluster[mask_hdb], labels_hdb[mask_hdb]) if mask_hdb.sum() > 1 else 0
db_hdb   = davies_bouldin_score(X_cluster[mask_hdb], labels_hdb[mask_hdb])   if mask_hdb.sum() > 1 else 999

df_comp = pd.DataFrame([
    {"Algoritmo": f"KMeans (k={best_k_km})",     "Clusters": best_k_km, "Ruido %": 0,
     "Silhouette ↑": round(sil_km,  4), "CH ↑": round(ch_km,  1), "DB ↓": round(db_km,  4)},
    {"Algoritmo": f"HDBSCAN (mcs={best_mcs})",   "Clusters": n_hdb,    "Ruido %": round(noise_hdb*100, 1),
     "Silhouette ↑": round(sil_hdb, 4), "CH ↑": round(ch_hdb, 1), "DB ↓": round(db_hdb, 4)},
])
print(f"Algorithm comparison [{X_cluster_name}]:")
display(df_comp)

# UMAP with the 2 algorithms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (lbl, title) in zip(axes, [
    (lbl_km,     f"KMeans k={best_k_km}"),
    (labels_hdb, f"HDBSCAN mcs={best_mcs}"),
]):
    unique_lbl = sorted(set(lbl))
    colors     = sns.color_palette("tab10", len(unique_lbl))
    cmap       = {l: colors[i] for i, l in enumerate(unique_lbl)}
    c          = [cmap[l] for l in lbl]
    ax.scatter(df["umap_x"], df["umap_y"], c=c, s=1, alpha=0.4, rasterized=True)
    patches = [mpatches.Patch(color=cmap[l], label=f"{'Noise' if l==-1 else l}") for l in unique_lbl]
    ax.legend(handles=patches, fontsize=7, loc="upper right")
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f"UMAP — KMeans vs HDBSCAN comparison  [{X_cluster_name}]", fontsize=13)
plt.tight_layout()
plt.show()

## 7 · Final clustering and profiling

KMeans (k by consensus) is selected as the final model — it guarantees full coverage with no noise,
necessary to assign a cluster to any new user in production.

In [ ]:
ALGO_FINAL   = f"KMeans k={best_k_km}"
labels_final = lbl_km
model_final  = km_comp

df["demo_cluster"] = labels_final
N_CLUSTERS = df["demo_cluster"].nunique()

print(f"Selected algorithm: {ALGO_FINAL}")
print(f"Number of clusters: {N_CLUSTERS}")
print("\nCluster sizes:")
print(df["demo_cluster"].value_counts().sort_index().to_string())

In [ ]:
# Demographic profile of each cluster
profile_feats = ["age", "ipa_class", "mun_type", "distance_type",
                 "gender_enc", "tiene_coche_enc", "size_hogar_enc",
                 "labor_status_enc", "civil_status_enc", "num_room_enc"]

profile = df.groupby("demo_cluster")[profile_feats].mean().round(2)
profile["n_usuarios"] = df["demo_cluster"].value_counts().sort_index()
print("Demographic profile by cluster:")
display(profile)

In [ ]:
# Boxplots of continuous features by cluster
cont_feats  = ["age", "ipa_class", "mun_type", "distance_type", "size_hogar_enc"]
cont_labels = ["Age", "IPA class", "Municipality type", "Dist. to capital", "Household size"]

fig, axes = plt.subplots(1, len(cont_feats), figsize=(18, 5))
colors = sns.color_palette("tab10", N_CLUSTERS)

for ax, feat, lbl in zip(axes, cont_feats, cont_labels):
    data = [df.loc[df["demo_cluster"] == c, feat].dropna().values
            for c in range(N_CLUSTERS)]
    bp = ax.boxplot(data, patch_artist=True, notch=False)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(lbl, fontsize=10)
    ax.set_xlabel("Cluster")
    ax.set_xticklabels(range(N_CLUSTERS))

plt.suptitle("Distribution of features by demographic cluster", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Binary / categorical variables by cluster
bin_feats  = ["gender_enc", "tiene_coche_enc", "labor_status_enc", "civil_status_enc"]
bin_labels = ["% Male", "% Owns a car", "Mean labour status", "Mean marital status"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = sns.color_palette("tab10", N_CLUSTERS)

for ax, feat, lbl in zip(axes, bin_feats, bin_labels):
    vals = df.groupby("demo_cluster")[feat].mean()
    ax.bar(vals.index, vals.values, color=colors[:len(vals)], edgecolor="white")
    ax.set_title(lbl, fontsize=10)
    ax.set_xlabel("Cluster")
    ax.set_ylim(0, max(vals.values) * 1.2)

plt.suptitle("Binary / categorical variables by cluster", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# UMAP coloured by final cluster
colors = sns.color_palette("tab10", N_CLUSTERS)
c_map  = [colors[c] for c in df["demo_cluster"]]

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(df["umap_x"], df["umap_y"], c=c_map, s=2, alpha=0.5, rasterized=True)
patches = [mpatches.Patch(color=colors[i], label=f"Cluster {i}") for i in range(N_CLUSTERS)]
ax.legend(handles=patches, title="Demo cluster", loc="upper right")
ax.set_title(f"UMAP — Demographic clustering ({ALGO_FINAL})", fontsize=13)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette plot by cluster
sil_samples = silhouette_samples(X_cluster, labels_final)
df["silhouette"] = sil_samples

fig, ax = plt.subplots(figsize=(8, 5))
y_lower = 10
colors  = sns.color_palette("tab10", N_CLUSTERS)

for i in range(N_CLUSTERS):
    vals = sil_samples[labels_final == i]
    vals.sort()
    y_upper = y_lower + len(vals)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals,
                     facecolor=colors[i], alpha=0.8)
    ax.text(-0.05, y_lower + 0.5 * len(vals), str(i), fontsize=9)
    y_lower = y_upper + 10

ax.axvline(sil_samples.mean(), color="crimson", linestyle="--",
           label=f"Mean={sil_samples.mean():.3f}")
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Cluster")
ax.set_title(f"Silhouette plot by cluster  [{X_cluster_name}]")
ax.legend()
plt.tight_layout()
plt.show()

print("Mean silhouette by cluster:")
print(df.groupby("demo_cluster")["silhouette"].mean().round(4).to_string())

## 8 · Interpretation and labelling of clusters

A descriptive label is assigned to each cluster based on its most distinctive features.

In [ ]:
# Most distinctive characteristics of each cluster (z-score relative to the global mean)
global_means = df[profile_feats].mean()
global_stds  = df[profile_feats].std().replace(0, 1)

print("Most distinctive characteristics by cluster (z-score):")
for c in range(N_CLUSTERS):
    cluster_mean = df[df["demo_cluster"] == c][profile_feats].mean()
    z = ((cluster_mean - global_means) / global_stds).sort_values(key=abs, ascending=False)
    top3 = z.head(3)
    n = (df["demo_cluster"] == c).sum()
    desc = ", ".join([f"{f}({v:+.2f}σ)" for f, v in top3.items()])
    print(f"  Cluster {c} (n={n:,}): {desc}")

In [ ]:
# Descriptive labels — assigned after reviewing the z-score above
CLUSTER_LABELS = {i: f"Cluster {i}" for i in range(N_CLUSTERS)}

for c in range(N_CLUSTERS):
    cluster_mean = df[df["demo_cluster"] == c][profile_feats].mean()
    z = ((cluster_mean - global_means) / global_stds)

    traits = []
    if z["num_room_enc"] > 1.0:            traits.append("Large dwellings (7+ rooms)")
    elif z["num_room_enc"] < -0.3:         traits.append("Standard dwellings (3-6 rooms)")
    if z["civil_status_enc"] > 0.5:        traits.append("Married")
    elif z["civil_status_enc"] < -0.5:     traits.append("Single")
    if z["age"] > 0.5:                     traits.append("Older")
    elif z["age"] < -0.5:                  traits.append("Young")
    if z["tiene_coche_enc"] > 0.5:         traits.append("With car")
    elif z["tiene_coche_enc"] < -0.5:      traits.append("Without car")
    if z["ipa_class"] > 0.5:               traits.append("High income")
    elif z["ipa_class"] < -0.5:            traits.append("Low income")
    if z["mun_type"] < -0.5:               traits.append("Small municipality")
    elif z["mun_type"] > 0.5:              traits.append("Metropolis")
    if z["size_hogar_enc"] > 0.5:          traits.append("Large household")
    elif z["size_hogar_enc"] < -0.5:       traits.append("Small household")
    if z["labor_status_enc"] > 0.5:        traits.append("Employed")
    elif z["labor_status_enc"] < -0.5:     traits.append("Inactive")
    if z["distance_type"] > 0.5:           traits.append("Rural/remote area")
    elif z["distance_type"] < -0.5:        traits.append("Nearby urban area")

    label = " · ".join(traits[:3]) if traits else f"Cluster {c}"
    CLUSTER_LABELS[c] = label
    print(f"  Cluster {c} (n={(df['demo_cluster']==c).sum():,}): {label}")

df["demo_cluster_label"] = df["demo_cluster"].map(CLUSTER_LABELS)

## 9 · Affinity demographic cluster × product category

The `demo_cluster` is joined with the historical events to compute the click rate
of each cluster on each product category. This matrix is the heart
of the cold-start recommendations in M3.

In [ ]:
prop = pd.read_csv(PROCESSED_PATH / "propensity_scores.csv")

prop_demo = prop.merge(
    df[["id_user", "demo_cluster", "demo_cluster_label"]], on="id_user", how="left"
)

aff_raw = (
    prop_demo.dropna(subset=["demo_cluster"])
    .groupby(["demo_cluster", "product_new"])["target"]
    .agg(n_obs="count", n_clicks="sum")
    .reset_index()
)
aff_raw["click_rate"] = aff_raw["n_clicks"] / aff_raw["n_obs"]

ALL_PRODUCTS = sorted(prop["product_new"].dropna().unique())

aff_matrix = (
    aff_raw.pivot(index="demo_cluster", columns="product_new", values="click_rate")
    .reindex(columns=ALL_PRODUCTS, fill_value=0)
    .fillna(0)
)
aff_norm = aff_matrix.div(aff_matrix.sum(axis=1), axis=0)

print("Affinity matrix:", aff_matrix.shape)
display(aff_matrix.round(3))

fig, ax = plt.subplots(figsize=(18, max(4, N_CLUSTERS)))
sns.heatmap(
    aff_matrix.T, cmap="YlOrRd", ax=ax, linewidths=0.3,
    annot=True, fmt=".2f", annot_kws={"size": 7},
    cbar_kws={"label": "Click rate"}
)
ax.set_title("Affinity demographic cluster × product category")
ax.set_xlabel("Demographic cluster")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
# Top-5 categories by cluster
print("Top-5 categories by demographic cluster:")
for c in sorted(aff_matrix.index):
    top5 = aff_matrix.loc[c].nlargest(5)
    n    = (df["demo_cluster"] == c).sum()
    lbl  = CLUSTER_LABELS[c]
    print(f"\n  Cluster {c} — {lbl} (n={n:,}):")
    for prod, cr in top5.items():
        print(f"    {prod:<30} cr={cr:.3f}")

In [ ]:
# Affinity by sector (more aggregated)
aff_sector = (
    prop_demo.dropna(subset=["demo_cluster"])
    .groupby(["demo_cluster", "sector"])["target"]
    .agg(n_obs="count", n_clicks="sum")
    .reset_index()
)
aff_sector["click_rate"] = aff_sector["n_clicks"] / aff_sector["n_obs"]

aff_sector_matrix = (
    aff_sector.pivot(index="demo_cluster", columns="sector", values="click_rate")
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(14, max(4, N_CLUSTERS)))
sns.heatmap(
    aff_sector_matrix.T, cmap="YlOrRd", ax=ax, linewidths=0.5,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    cbar_kws={"label": "Click rate"}
)
ax.set_title("Affinity demographic cluster × sector")
ax.set_xlabel("Demographic cluster")
plt.tight_layout()
plt.show()

## 10 · Export

In [ ]:
import pickle

# Cluster assignment
df_out = df[["id_user", "demo_cluster", "demo_cluster_label"]].copy()
df_out.to_csv(PROCESSED_PATH / "users_demo_segments.csv", index=False)

# Affinity matrices
aff_matrix.to_csv(PROCESSED_PATH / "demo_cluster_affinity.csv")
aff_norm.to_csv(PROCESSED_PATH   / "demo_cluster_affinity_norm.csv")

# Scaler and clustering model (needed for cold-start in M3)
with open(PROCESSED_PATH / "demo_scaler.pkl", "wb") as f: pickle.dump(scaler, f)
with open(PROCESSED_PATH / "demo_model.pkl",  "wb") as f: pickle.dump(model_final, f)

# Flags for the inference pipeline
with open(PROCESSED_PATH / "demo_ae_used.pkl",  "wb") as f: pickle.dump(AE_USED,  f)
with open(PROCESSED_PATH / "demo_pca_used.pkl", "wb") as f: pickle.dump(PCA_USED, f)

if AE_USED:
    encoder.save(str(PROCESSED_PATH / "demo_encoder.keras"))
    print(f"Exported: demo_encoder.keras  ({INPUT_DIM}D → {LATENT_DIM}D)")
if PCA_USED:
    with open(PROCESSED_PATH / "demo_pca.pkl", "wb") as f: pickle.dump(pca, f)
    print(f"Exported: demo_pca.pkl  (PCA {LATENT_DIM}D, variance={pca.explained_variance_ratio_.sum():.1%})")

print(f"Exported: users_demo_segments.csv        {df_out.shape}")
print(f"Exported: demo_cluster_affinity.csv      {aff_matrix.shape}")
print(f"Exported: demo_cluster_affinity_norm.csv {aff_norm.shape}")
print(f"Exported: demo_scaler.pkl  (StandardScaler)")
print(f"Exported: demo_model.pkl   ({ALGO_FINAL})")
print(f"Clustering space: {X_cluster_name}  →  AE={AE_USED}  PCA={PCA_USED}")

display(df_out.head())
print("\nFinal distribution:")
print(df_out.groupby(["demo_cluster", "demo_cluster_label"])
            .size().reset_index(name="n_usuarios").to_string(index=False))

## Methodological decisions

| # | Decision | Justification |
|---|----------|--------------|
| D-28 | Demographic clustering independent of M1 | The M1 segments are behavioural; demographics alone only predict M1 with ~39% accuracy |
| D-29 | Features: the same 10 as M2 | Pipeline coherence: any new user reaching M2 can also be segmented here |
| D-30 | Algorithms evaluated: KMeans, GMM, HDBSCAN | Coverage of the three paradigms (partition, probabilistic, density) |
| D-31 | Final selection by silhouette (excluding noise) | HDBSCAN can produce high noise with demographic data; KMeans/GMM guarantee full assignment |
| D-32 | Outputs: `users_demo_segments.csv`, affinity matrices | Separation of responsibilities: the notebook only does segmentation; M3 consumes the result |

**Use in production (cold-start):**
```python
# For a new user:
x_scaled  = scaler.transform([features_usuario])
cluster   = km_demo.predict(x_scaled)[0]
top5      = aff_norm.loc[cluster].nlargest(5)
```